# 03. Model Training - MobileNetV2 Transfer Learning

- Phase 1: Backbone freeze, classification head만 학습
- Phase 2: Fine-tuning (상위 레이어 unfreeze)
- 학습 결과 시각화

In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
import json
import os
import matplotlib.pyplot as plt

IMG_SIZE = 224
BATCH_SIZE = 32
DATA_DIR = '/Users/parkyoungbin/Desktop/ml2/model/data'
SAVE_DIR = '/Users/parkyoungbin/Desktop/ml2/model/saved_model'

# 데이터 로드
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
val_df = pd.read_csv(os.path.join(DATA_DIR, 'val.csv'))

labels_sorted = sorted(train_df['label'].unique())
label2idx = {l: i for i, l in enumerate(labels_sorted)}
NUM_CLASSES = len(labels_sorted)
train_df['label_idx'] = train_df['label'].map(label2idx)
val_df['label_idx'] = val_df['label'].map(label2idx)

with open(os.path.join(DATA_DIR, 'class_weights.json')) as f:
    class_weights = {int(k): v for k, v in json.load(f).items()}

print(f'Classes: {labels_sorted}')
print(f'Train: {len(train_df)} | Val: {len(val_df)}')

In [ ]:
# tf.data 파이프라인
def load_and_preprocess(image_path, label):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.keras.applications.mobilenet_v2.preprocess_input(img)
    return img, label

def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, 0.15)
    image = tf.image.random_contrast(image, 0.8, 1.2)
    image = tf.image.random_crop(
        tf.image.resize(image, [IMG_SIZE + 20, IMG_SIZE + 20]),
        [IMG_SIZE, IMG_SIZE, 3]
    )
    return image, label

def make_dataset(dataframe, training=False):
    paths = dataframe['image_path'].values
    labels = tf.one_hot(dataframe['label_idx'].values, NUM_CLASSES)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.shuffle(2048)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df, training=False)
print('Dataset 준비 완료')

## Phase 1: Backbone Freeze + Head 학습

In [ ]:
# MobileNetV2 backbone
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

# Classification head
model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
callbacks_p1 = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(
        os.path.join(SAVE_DIR, 'fashion_phase1_best.keras'),
        monitor='val_accuracy', save_best_only=True, verbose=1
    )
]

print('Phase 1 학습 시작 (backbone frozen)...')
history_p1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weights,
    callbacks=callbacks_p1
)
print(f'\nPhase 1 Best val_accuracy: {max(history_p1.history["val_accuracy"]):.4f}')

## Phase 2: Fine-tuning (상위 레이어 unfreeze)

In [ ]:
# 마지막 50개 레이어 unfreeze
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
print(f'Trainable layers in backbone: {trainable_count} / {len(base_model.layers)}')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
callbacks_p2 = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(
        os.path.join(SAVE_DIR, 'fashion_phase2_best.keras'),
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)
]

print('Phase 2 학습 시작 (fine-tuning)...')
history_p2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    class_weight=class_weights,
    callbacks=callbacks_p2
)
print(f'\nPhase 2 Best val_accuracy: {max(history_p2.history["val_accuracy"]):.4f}')

## 학습 결과 시각화

In [ ]:
# Phase 1 + Phase 2 히스토리 합치기
all_acc = history_p1.history['accuracy'] + history_p2.history['accuracy']
all_val_acc = history_p1.history['val_accuracy'] + history_p2.history['val_accuracy']
all_loss = history_p1.history['loss'] + history_p2.history['loss']
all_val_loss = history_p1.history['val_loss'] + history_p2.history['val_loss']
p1_epochs = len(history_p1.history['accuracy'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
ax1.plot(all_acc, label='Train Accuracy')
ax1.plot(all_val_acc, label='Val Accuracy')
ax1.axvline(x=p1_epochs - 0.5, color='gray', linestyle='--', alpha=0.5, label='Phase 2 start')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.set_title('Training & Validation Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Loss
ax2.plot(all_loss, label='Train Loss')
ax2.plot(all_val_loss, label='Val Loss')
ax2.axvline(x=p1_epochs - 0.5, color='gray', linestyle='--', alpha=0.5, label='Phase 2 start')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.set_title('Training & Validation Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'training_history.png'), dpi=150)
plt.show()
print('학습 그래프 저장 완료')

## 최종 모델 저장

In [ ]:
# 최종 모델 저장
model.save(os.path.join(SAVE_DIR, 'fashion_final.keras'))

# 학습 기록 저장
results = {
    'phase1_best_val_acc': float(max(history_p1.history['val_accuracy'])),
    'phase2_best_val_acc': float(max(history_p2.history['val_accuracy'])),
    'phase1_epochs': len(history_p1.history['accuracy']),
    'phase2_epochs': len(history_p2.history['accuracy']),
    'total_epochs': len(all_acc),
    'final_train_acc': float(all_acc[-1]),
    'final_val_acc': float(all_val_acc[-1]),
    'classes': labels_sorted,
    'num_classes': NUM_CLASSES,
}

with open(os.path.join(SAVE_DIR, 'training_results.json'), 'w') as f:
    json.dump(results, f, indent=2)

print('=== 학습 완료 ===')
print(f'Phase 1 best val_acc: {results["phase1_best_val_acc"]:.4f}')
print(f'Phase 2 best val_acc: {results["phase2_best_val_acc"]:.4f}')
print(f'Total epochs: {results["total_epochs"]}')
print(f'\n저장 경로: {SAVE_DIR}/fashion_final.keras')